# Agent-2: IBM Quantum Process Tomography

This notebook reconstructs Choi matrices for gate processes using quantum process tomography (QPT).  It is designed to run offline by default, while documenting the separate IBM Quantum hardware submission/retrieval path.

Shared palette: `#2F4858`, `#33658A`, `#86BBD8`, `#F6AE2D`, `#F26419`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

from qpt_tools import (
    amplitude_damping_after_unitary,
    choi_from_unitary,
    depolarizing_after_unitary,
    diagnose_noise,
    is_cp,
    is_tp,
    linear_inversion_choi,
    matrix_to_json_dict,
    mle_choi,
    plot_bloch_deformation,
    plot_choi_heatmap,
    save_json,
    simulate_output_states_from_choi,
    two_qubit_depolarizing_after_unitary,
)

np.random.seed(42)
PALETTE = ["#2F4858", "#33658A", "#86BBD8", "#F6AE2D", "#F26419"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=PALETTE)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 1. QPT Theory Primer

For a channel $\mathcal{E}$, process tomography is state tomography on the unnormalized Choi matrix

$$C_{\mathcal{E}} = \sum_{ij} |i\rangle\langle j|_A \otimes \mathcal{E}(|i\rangle\langle j|)_B,$$

with input system $A$ first and output system $B$ second.  Trace preservation is $\mathrm{Tr}_B(C_{\mathcal{E}})=I_A$.

For one qubit, the four inputs `|0>`, `|1>`, `|+>`, and `|+i>` are informationally complete.  Linear inversion reconstructs the four output blocks directly, but finite-shot data can make the result non-physical.  The MLE step used here is the nearest CPTP projection: positive semidefinite Choi matrix and `Tr_B(C) = I_A`.


## 2. Implementation

The helper module `qpt_tools.py` provides:

- `run_process_tomography` for optional Qiskit Experiments submission.
- `linear_inversion_choi` for exact/simulated one-qubit QPT data.
- `mle_choi` for CVXPY/SCS CPTP projection, with an offline alternating-projection fallback.
- Choi fidelity, average gate fidelity, Kraus weights, and heuristic noise diagnosis.

The default run below analyzes X, H, and CNOT using simulated noisy processes so the notebook is deterministic without IBM credentials.

In [ ]:
X = np.array([[0, 1], [1, 0]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
CNOT = np.array(
    [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]],
    dtype=complex,
)

targets = {
    "X": {
        "dimension": 2,
        "ideal": choi_from_unitary(X),
        "actual": amplitude_damping_after_unitary(X, gamma=0.08),
        "noise_model": "amplitude_damping_after_unitary(gamma=0.08)",
    },
    "H": {
        "dimension": 2,
        "ideal": choi_from_unitary(H),
        "actual": depolarizing_after_unitary(H, p=0.06),
        "noise_model": "depolarizing_after_unitary(p=0.06)",
    },
    "CNOT": {
        "dimension": 4,
        "ideal": choi_from_unitary(CNOT),
        "actual": two_qubit_depolarizing_after_unitary(CNOT, p=0.08),
        "noise_model": "two_qubit_depolarizing_after_unitary(p=0.08)",
    },
}

## 3. Experimental Run

Mode used in this notebook: offline simulated fallback.  The shot count is recorded as 4096 to match the planned hardware workflow; the simulated matrices here are deterministic exact expectations rather than sampled counts.

In [ ]:
raw_results = {
    "mode": "offline_simulated",
    "seed": 42,
    "shots_per_circuit": 4096,
    "qiskit_hardware_job_ids": [],
    "gates": {},
}
reconstructed = {}

for gate, spec in targets.items():
    d = spec["dimension"]
    if d == 2:
        measurement_data = {"output_states": simulate_output_states_from_choi(spec["actual"])}
        linear = linear_inversion_choi(measurement_data)
        mle = mle_choi(measurement_data, d_in=2, d_out=2)
        raw_measurement = {
            label: matrix_to_json_dict(state)
            for label, state in measurement_data["output_states"].items()
        }
    else:
        measurement_data = {"choi": spec["actual"]}
        linear = linear_inversion_choi(measurement_data)
        mle = mle_choi(measurement_data, d_in=4, d_out=4)
        raw_measurement = {"linear_inversion_choi": matrix_to_json_dict(linear)}

    diagnosis = diagnose_noise(mle, spec["ideal"])
    reconstructed[gate] = {"linear": linear, "mle": mle, "diagnosis": diagnosis}
    raw_results["gates"][gate] = {
        "dimension": d,
        "noise_model": spec["noise_model"],
        "measurement_data": raw_measurement,
        "ideal_choi": matrix_to_json_dict(spec["ideal"]),
        "reconstructed_choi": matrix_to_json_dict(mle),
        "diagnosis": diagnosis,
    }

save_json(raw_results, DATA_DIR / "raw_results.json")

Hardware submission and retrieval should be run separately so a queued IBM job does not block the notebook:

```python
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qpt_tools import run_process_tomography, save_json

service = QiskitRuntimeService()
backend = service.backend("ibm_brisbane")

qc = QuantumCircuit(1)
qc.x(0)
result = run_process_tomography(qc, backend, shots=4096)
save_json({"metadata": result.metadata}, "data/hardware_job_metadata.json")

# Later, retrieve by saved job IDs with the same IBM Runtime account and
# re-run local Qiskit Experiments analysis when the jobs are complete.
```

For a realistic simulator fallback, replace the backend with `AerSimulator.from_backend(FakeBrisbane())` when the installed Qiskit version provides the fake backend package.

## 4. Choi Matrix Reconstruction

The table compares ideal, linear inversion, and MLE-projected Choi diagnostics.  Exact simulated one-qubit data are already physical, but finite-shot or otherwise perturbed linear-inversion estimates need not be.  The short perturbation demo below creates a non-physical linear-inversion estimate and then projects it back to the CPTP set with MLE.


In [ ]:
summary_rows = []
for gate, spec in targets.items():
    diag = reconstructed[gate]["diagnosis"]
    summary_rows.append(
        (
            gate,
            diag["process_fidelity"],
            diag["average_gate_fidelity"],
            diag["is_cp"],
            diag["is_tp"],
            diag["dominant_noise"],
        )
    )

print("gate   F_pro      F_avg      CP    TP    diagnosis")
for row in summary_rows:
    print(f"{row[0]:<5}  {row[1]:.6f}  {row[2]:.6f}  {row[3]!s:<5} {row[4]!s:<5} {row[5]}")

In [ ]:
nonphysical_linear = reconstructed["X"]["linear"].copy()
nonphysical_linear[0, 0] -= 0.35
nonphysical_linear[3, 3] += 0.15
nonphysical_linear = 0.5 * (nonphysical_linear + nonphysical_linear.conj().T)
repaired_mle = mle_choi({"choi": nonphysical_linear}, d_in=2, d_out=2)

print("Perturbed linear inversion:")
print("  min eigenvalue:", f"{np.min(np.linalg.eigvalsh(nonphysical_linear)):.6f}")
print("  CP / TP:", is_cp(nonphysical_linear), is_tp(nonphysical_linear, d_in=2, d_out=2))
print("After MLE projection:")
print("  min eigenvalue:", f"{np.min(np.linalg.eigvalsh(repaired_mle)):.6f}")
print("  CP / TP:", is_cp(repaired_mle, tol=5e-6), is_tp(repaired_mle, d_in=2, d_out=2, tol=5e-6))


## 5. Diagnosis

The one-qubit diagnosis inspects the residual Pauli-transfer action after factoring out the ideal gate.  Translation of the Bloch sphere indicates relaxation-like noise; isotropic shrinkage indicates depolarization; axis-dependent shrinkage indicates dephasing or Pauli-biased noise.  For CNOT, the sample uses a global two-qubit depolarizing model, visible as one dominant Kraus weight and many small equal weights.

In [ ]:
for gate, spec in targets.items():
    diag = reconstructed[gate]["diagnosis"]
    print(f"\n{gate}: {diag['dominant_noise']}")
    print(f"  process fidelity: {diag['process_fidelity']:.6f}")
    print(f"  average gate fidelity: {diag['average_gate_fidelity']:.6f}")
    print(f"  half-diamond distance (SDP): {diag['diamond_distance']:.6f}")
    print(f"  diamond-distance proxy (Choi nuclear norm): {diag['diamond_distance_proxy']:.6f}")
    print(f"  leading Kraus weights: {[round(w, 6) for w in diag['kraus_weights'][:5]]}")

## 6. Visualization

The Choi heatmaps show ideal versus reconstructed structure.  The Bloch deformation plots are shown for one-qubit gates only.

In [ ]:
for gate in ["X", "H", "CNOT"]:
    plot_choi_heatmap(targets[gate]["ideal"], f"{gate} ideal Choi")
    plot_choi_heatmap(reconstructed[gate]["mle"], f"{gate} reconstructed Choi")

In [ ]:
plot_bloch_deformation(reconstructed["X"]["mle"])
plot_bloch_deformation(reconstructed["H"]["mle"])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(summary_rows))
ax.bar(x - 0.18, [row[1] for row in summary_rows], width=0.36, label="process fidelity")
ax.bar(x + 0.18, [row[2] for row in summary_rows], width=0.36, label="average gate fidelity")
ax.set_xticks(x, [row[0] for row in summary_rows])
ax.set_ylim(0, 1.05)
ax.set_ylabel("fidelity")
ax.set_title("Ideal vs reconstructed process quality")
ax.legend()
plt.show()

## Summary

This offline fallback reproduces the full Choi-analysis path for at least three gates.  Real IBM data can be dropped into `data/raw_results.json` using the same schema: raw measurement payload, reconstructed Choi matrix, job IDs, and diagnostics.  The simulated results are not hardware claims; they are deterministic fixtures for development and validation.  Diamond norm language is reserved for the SDP-computed half-diamond distance; the nuclear-norm quantity is reported only as a clearly labeled proxy.
